## Context

The project must accept local medical documents in these forms:

- Native-text PDF
- Scanned PDF
- JPEG/JPG image
- PNG image
- Handwritten-note image

The resulting text must be divided into retrievable chunks while preservingdocument, page, source format, extraction method, patient, section, and sourceoffset information.

The supplied synthetic PDF is a one-page medical record containing patient IDSYN-200989, medical sections, medication, allergy, vital-sign, laboratory,and imaging information. It is used as the initial local PDF example.

## Decision

Day 2 will include a small local ingestion layer insideapp/ingestion/chunking.py so the learning milestone can be exercised end-to-end:

Local file or directory

        ↓
Format discovery and validation

        ↓
Native PDF extraction or OCR

        ↓
DocumentPage models

        ↓
Section detection

        ↓
Page-bounded chunks with overlap


Supported suffixes are:

- .pdf
- .jpeg
- .jpg
- .png

Image files whose names contain handwritten or handwriting are marked ashandwritten notes. They use the same OCR engine but retain a distinctOCR_HANDWRITTEN extraction method and is_handwritten=true metadata.

## PDF behavior

1. Attempt native text extraction with PyMuPDF.
2. Count meaningful alphanumeric characters.
3. If native text is below the configured threshold, render the page.
4. OCR the rendered page with Tesseract through pytesseract.
5. Preserve the original page number.

This avoids OCRing every PDF page unnecessarily.

### Image and handwritten-note behavior

- Open the local image with Pillow.
- Apply EXIF orientation.
- Convert it to grayscale.
- Apply automatic contrast.
- Run Tesseract OCR.
- Treat the image as page 1.
- Mark handwritten notes separately when requested or inferred from filename.

Handwriting recognition quality varies significantly by handwriting,resolution, lighting, orientation, and OCR model. Tesseract support here is abaseline, not a claim of production-grade handwriting recognition.

### Chunking decision

Chunks are:

- Bound to one page
- Bound to one source document
- Bound to one patient metadata context
- Section-aware
- Character-limited
- Overlapped only within a section
- Traceable by exact character offsets
- Assigned deterministic IDs

The initial configuration remains:
- max_characters: 800
- overlap_characters: 150
- minimum_chunk_characters: 1

These are experimental defaults.

## Why keep ingestion and chunking together on Day 2?

The long-term architecture should separate:

- file discovery
- PDF extraction
- image preprocessing
- OCR
- metadata extraction
- section detection
- chunking

For the Day 2 learning milestone, keeping the minimal local ingestion entrypoints next to chunking makes the deliverable runnable without introducing additional requested files. Later days should refactor extraction and OCR into their own modules.

## Consequences

### Positive
- A local file or directory can be ingested directly.
- Native PDF text avoids unnecessary OCR.
- Scanned PDFs receive an OCR fallback.
- JPEG/JPG/PNG files are supported.
- Handwritten notes retain explicit metadata.
- The provided synthetic PDF is covered by a test.
- Page-level citations remain possible.

## Limitations

- Tesseract must be installed separately.
- Tesseract may perform poorly on difficult handwriting.
- Heading detection is heuristic.
- Tables and multi-column layouts may need layout-aware extraction.
- Character limits do not equal token limits.
- The current handwritten flag relies partly on naming convention unless thecaller sets it explicitly.

## Validation

The unit suite checks:

- Local source validation
- File discovery
- PDF/image format recognition
- Handwritten-note detection
- Image OCR through a deterministic test double
- Patient ID extraction
- Section detection
- Overlap
- Offset reconstruction
- The supplied SYN-200989.pdf example